In [1]:
# !pip install transformers datasets pandas torch accelerate bitsandbytes

import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from huggingface_hub import login

# load in CVS of emails
df = pd.read_csv('../data/raw/sample_emails.csv')

# check if loaded
print(f"Loaded {len(df)} emails")
df.head

#Checking if cuda is avilable
print(torch.cuda.is_available())

Loaded 10 emails
True


In [2]:
login()

# define modle
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it")


# Load the text-generation pipeline
# device="cuda" if you havbe a GPU, otherwise "cpu"

pipe = pipeline(
    "text-generation",
    tokenizer=tokenizer,
    model=model,
    device="cuda",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

print(f'Loaded {model}')

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [3]:
# n.b Zero shot is no example in the prompt, just instructions (useful video - https://youtu.be/QRsngDNp3cY)
def build_classification_prompt_zero_shot(subject, body):
    """
    Build prompt asking the model to classify an email into one of 5 categories.
    """
    prompt = f"""You are an AI Assistant helping an insurance company categorize customer emails.
    
    Categories:
    - claim: customer wants to make and insurance claim
    - complaint: customer is unhappy and wants to raise a complaint
    - renewal: customer has questions about policy renewal
    - policy_change: customer wants to change details on their policy
    - general_query: general questions or requests

    Email Subject: {subject}
    Email Body: {body}

    Which category does this email belong to? Answer with only the category name (claim, complaint, renewal, policy_change, or general query).

    Category:"""

    return prompt

# Test on the first email
test_email = df.iloc[0]
prompt = build_classification_prompt_zero_shot(test_email['email_subject'], test_email['email_body'])
print(prompt)

You are an AI Assistant helping an insurance company categorize customer emails.

    Categories:
    - claim: customer wants to make and insurance claim
    - complaint: customer is unhappy and wants to raise a complaint
    - renewal: customer has questions about policy renewal
    - policy_change: customer wants to change details on their policy
    - general_query: general questions or requests

    Email Subject: Car accident claim
    Email Body: Hi, I was involved in a minor car accident yesterday. No one was injured but my rear bumper is badly damaged. I have a comprehensive policy with you. Can you tell me how to start a claim and whether I need to pay an excess?

    Which category does this email belong to? Answer with only the category name (claim, complaint, renewal, policy_change, or general query).

    Category:


In [4]:
# Generate a response using the pipeline
output = pipe(
    prompt,
    max_new_tokens=10, # 10 should be enough, 1-2 tokens for the category name
    do_sample=False, # this property will provide a deterministic output, same input -> same output
    return_full_text=False # only return generate part, no prompt return
)

# Extract the generate text
predicited_category = output[0]['generated_text'].strip().lower()

print(f"True category: {test_email['category']}")
print(f"Predicted category: {predicited_category}")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


True category: claim
Predicted category: **claim** 


**explanation:**

the


In [5]:
# Now we will run on all emails with zero shot

def classify_email_zero_shot():
    """Run classification and return just the predicited category"""
    prompts = [build_classification_prompt_zero_shot(row['email_subject'], row['email_body'])
              for _, row in df.iterrows()]
    outputs = pipe(prompts, max_new_tokens=10, do_sample=False, return_full_text=False, batch_size=64)
 
    # Extract first word
    predictions = [ output[0]['generated_text'].strip().split()[0].lower().replace('**','')
                                 for output in outputs]

    return predictions


df['predicited_zero_shot'] = classify_email_zero_shot()

# Calculate accuracy
correct = (df['category'] == df['predicited_zero_shot']).sum()
accuracy = correct / len(df)

print(f"Zero-shot accuracy: {correct}/{len(df)} = {accuracy:.2%}")
print(f"\nPredictions")
print(df[['id', 'category', 'predicited_zero_shot']])

Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zero-shot accuracy: 7/10 = 70.00%

Predictions
   id       category predicited_zero_shot
0   1          claim                claim
1   2          claim                claim
2   3      complaint            complaint
3   4      complaint            complaint
4   5        renewal              renewal
5   6        renewal        general_query
6   7  policy_change        general_query
7   8  policy_change        general_query
8   9  general_query        general_query
9  10  general_query        general_query


## Issues noted
We only have a 70% accuracy with email 7 & 8 being clasified as gernal queries, this is potentially due to ambigious question asked, the customer is asking about a change to the policy but not specifically saying that they want to change it.

In [6]:
# Few-shot -> include 2-3 labeled examples in the propmt for guidence
def build_classification_prompt_few_shot(subject, body):
    """
    Few-shot prompt with 3 examples emails before the test email.
    """

    prompt = f"""You are an AI assistant helping an insurance company categorize customer emails.

Categories:
    - claim: customer wants to make and insurance claim
    - complaint: customer is unhappy and wants to raise a complaint
    - renewal: customer has questions about policy renewal
    - policy_change: customer wants to change details on their policy
    - general_query: general questions or requests

Examples:
- Example 1:
    Email Subject: Car accident claim
    Email Body: Hi, I was involved in a minor car accident yesterday. Can you tell me how to start a claim and any other information I need to provide?
    Category: claim

- Example 2:
    Email Subject: Unhappy with claim decision
    Email Body: I recently submitted a claim and received a decision saying the items are not covered. I am very unhappy with this.
    Category: complaint

- Example 3:
    Email Subject: Renewal Query & Auto-renewal opt out
    Email Body: Hi, I think my car insurance is due for renewal soon but I can't find the email. When does my policy end? and How would I turn off auto-renewal so that policy will end on the renewal date?
    Category: renewal

- Example 4:
    Email Subject: No claims discount proof
    Email Body: I am switching to a different insurer and they have asked for proof of my no claims discount. Can you give me details how I would request this?
    Category: general_query

- Example 5:
    Email Subject: Adding a named driver & change of address
    Email Body: Hi, I would like to provide an address update and add a driver to my policy. Coule you let me know what information I would need to provide and how likely my premiums are to go up?
    Category: policy_change

Now classify this email:

Email Subject: {subject}
Email Body: {body}

Taking the above examples into consideration which category out of the list above would you categorize this email with? Pelase only reply with the category of the email (claim, complaint, renewal, policy_change, or general query) and strictly nothing else.

Category:"""

    return prompt

test_email = df.iloc[5] # Using number 7 as last Zero Shot was incorrect for this one
prompt = build_classification_prompt_few_shot(test_email['email_subject'], test_email['email_body'])
output = pipe(prompt, max_new_tokens=10, do_sample=False, return_full_text=False)
print(f"Predicted: {output[0]['generated_text'].strip()}")
print(f"True: {test_email['category']}")


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predicted: renewal
True: renewal


In [7]:
def classify_email_few_shot():
    """Run classification and return just the predicited category"""
    prompts = [build_classification_prompt_few_shot(row['email_subject'], row['email_body'])
              for _, row in df.iterrows()]
    outputs = pipe(prompts, max_new_tokens=10, do_sample=False, return_full_text=False, batch_size=64)
 
    # Extract first word
    predictions = [ output[0]['generated_text'].strip().split()[0].lower().replace("**","")
                                 for output in outputs]

    return predictions


df['predicited_few_shot'] = classify_email_few_shot()

# Calculate accuracy
correct = (df['category'] == df['predicited_few_shot']).sum()
accuracy = correct / len(df)

print(f"Few accuracy: {correct}/{len(df)} = {accuracy:.2%}")
print(f"\nPredictions")
print(df[['id', 'category', 'predicited_few_shot']])

Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Few accuracy: 10/10 = 100.00%

Predictions
   id       category predicited_few_shot
0   1          claim               claim
1   2          claim               claim
2   3      complaint           complaint
3   4      complaint           complaint
4   5        renewal             renewal
5   6        renewal             renewal
6   7  policy_change       policy_change
7   8  policy_change       policy_change
8   9  general_query       general_query
9  10  general_query       general_query


In [ ]:
def build_reply_prompt(subject, body, category):
    """
    Prompt to draft a professional insurance reply email
    """

    prompt = f"""You are a customer service agent at a UK motor and home insurance company. Write a polite, professional, and helpful reply to the following customer email.

    Customer Email:
    Subject: {subject}
    Body: {body}

    This email is a: {category}

    Guideline:
    - Be polite and empathetic
    - Use UK English
    - Keep it concise (3-4 short paragraphs)
    - Sign off with "Kind regards, Ryan Sylvester, Customer Services"

    Reply:"""
    return prompt

def build_reply_prompt_v2(subject, body, category):
    """
    Prompt to draft a professional insurance reply email
    """

    prompt = f"""You are a UK insurance customer service agent. Write a reply following this exact structure:

    - Greeting and acknowledgement
    - Explanation or answer to their questions
    - Next steps or actions
    - Closing with "Kind regards, Ryan Sylvester, Customer Services"

    Customer Email (category: {category}):
    Subject: {subject}
    Body: {body}

    Reply:"""
    return prompt

# Test on first email (Claim)
test_email = df.iloc[0]
prompt_v1 = build_reply_prompt(
    test_email['email_subject'],
    test_email['email_body'],
    test_email['category']
)
prompt_v2 = build_reply_prompt_v2(
    test_email['email_subject'],
    test_email['email_body'],
    test_email['category']
)

print("VERSION 1 OUTPUT")
print(pipe(prompt_v1, max_new_tokens=200,do_sample=True,temperature=0.7,return_full_text=False)[0]['generated_text'])
print("\n" + "="*50 + "\n")
print("VERSION 2 OUTPUT")
print(pipe(prompt_v2, max_new_tokens=200,do_sample=True,temperature=0.7,return_full_text=False)[0]['generated_text'])


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


VERSION 1 OUTPUT
